## import packages

In [1]:
import numpy as np
import pandas as pd
from typing import Optional, Tuple
import sys

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from pickle import dump
from sklearn.preprocessing import MinMaxScaler
import time
from tqdm import tqdm

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import os

#device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("cuda is available")
else:
    print("cuda is NOT available")

#device = torch.device("cuda:0")


import shutil
import warnings
import pickle

warnings.filterwarnings("ignore")
import logging

logging.disable(logging.CRITICAL)

from nn_functions import surrogate
from moving_average import moving_average_1d
import copy
from GAMMA_obj_temp_depth import GAMMA_obj

import sys
sys.path.append('../1_model')
from TiDE import TideModule, quantile_loss, TiDE_forward

import importlib
import policy
importlib.reload(policy)
from policy import PolicyNN


cuda is available


In [2]:
import os, torch

# ===== 여기서 device 선택 =====
GPU_ID = 7          # 쓸 GPU 번호 (0,1,2,...). CPU로 쓰려면 -1

if GPU_ID >= 0 and torch.cuda.is_available():
    device = torch.device(f"cuda:{GPU_ID}")
    torch.cuda.set_device(GPU_ID)
    print(f"using device: {device} ({torch.cuda.get_device_name(GPU_ID)})")
else:
    device = torch.device("cpu")
    print("using device: cpu")

print("device_count =", torch.cuda.device_count())
print("current_device =", torch.cuda.current_device() if torch.cuda.is_available() else None)

using device: cuda:7 (NVIDIA RTX A6000)
device_count = 8
current_device = 7


## define funcitons

In [3]:
# values from user
x_min = torch.tensor([[0.0, 0.75, 0.75, 504.26]], dtype=torch.float32).to(device)
x_max = torch.tensor([[7.5, 20.0, 20.0, 732.298]], dtype=torch.float32).to(device)

y_min = torch.tensor([[436.608, -0.559]], dtype=torch.float32).to(device)
y_max = torch.tensor([[4509.855, 0.551]], dtype=torch.float32).to(device)


In [4]:
def normalize_x(x, dim_id):
    x_min_selected = x_min[0, dim_id]
    x_max_selected = x_max[0, dim_id]
    return 2 * (x - x_min_selected) / (x_max_selected - x_min_selected) - 1

def inverse_normalize_x(x_norm, dim_id):
    x_min_selected = x_min[0, dim_id]
    x_max_selected = x_max[0, dim_id]
    return 0.5 * (x_norm + 1) * (x_max_selected - x_min_selected) + x_min_selected

def normalize_y(y, dim_id):
    y_min_selected = y_min[0, dim_id]
    y_max_selected = y_max[0, dim_id]
    return 2 * (y - y_min_selected) / (y_max_selected - y_min_selected) - 1

def inverse_normalize_y(y_norm, dim_id):
    y_min_selected = y_min[0, dim_id]
    y_max_selected = y_max[0, dim_id]
    return 0.5 * (y_norm + 1) * (y_max_selected - y_min_selected) + y_min_selected


In [5]:
def run_one_step_policy(GAMMA_obj, policy_model, P, window):
    # Reference trajectory for temperature (original scale)
    mp_temp_ref = GAMMA_obj.ref[GAMMA_obj.MPC_counter:GAMMA_obj.MPC_counter + P]
    mp_temp_ref_t = torch.as_tensor(mp_temp_ref, dtype=torch.float32, device=device).reshape(1, P, 1)

    # Past input (original scale)
    mp_temp_past_t = GAMMA_obj.x_past.T.unsqueeze(0).to(device)
    laser_past_t = GAMMA_obj.u_past.view(1, -1, 1).to(device)
    fix_cov_past = GAMMA_obj.fix_cov_all[GAMMA_obj.MPC_counter - window:GAMMA_obj.MPC_counter, :]
    fix_cov_past_t = torch.as_tensor(fix_cov_past, dtype=torch.float32, device=device).unsqueeze(0)

    # Normalize
    fix_cov_past_s = normalize_x(fix_cov_past_t, dim_id=[0, 1, 2])
    laser_past_s = normalize_x(laser_past_t, dim_id=[3])
    mp_temp_past_s = normalize_y(mp_temp_past_t, dim_id=[0, 1])

    policy_in_past = torch.cat((fix_cov_past_s, laser_past_s, mp_temp_past_s), dim=2)

    # Future covariates
    fix_cov_future = GAMMA_obj.fix_cov_all[GAMMA_obj.MPC_counter:GAMMA_obj.MPC_counter + P, :]
    fix_cov_future_t = torch.as_tensor(fix_cov_future, dtype=torch.float32, device=device).unsqueeze(0)
    fix_cov_future_s = normalize_x(fix_cov_future_t, dim_id=[0, 1, 2])
    mp_temp_ref_s = normalize_y(mp_temp_ref_t, dim_id=[0])[:, :, 0].unsqueeze(-1)

    depth_lower_const = 0.1423
    depth_upper_const = 0.4126
    y_const_s = torch.tensor([[depth_lower_const, depth_upper_const]] * P, dtype=torch.float32, device=device).reshape(1, P, 2)

    policy_in_future = torch.cat((fix_cov_future_s, mp_temp_ref_s, y_const_s), dim=2)

    # ✅ Policy inference 시간 측정 (순수 forward만)
    t1 = time.time()
    u_pred = policy_model((policy_in_past, policy_in_future))
    t2 = time.time()
    GAMMA_obj.save_time.append(t2 - t1)

    # 현재 예측된 첫 control (normalized 상태)
    u_first = u_pred[0, 0]

    # 이전 control
    u_prev = normalize_x(GAMMA_obj.u_past[-1].view(1,1,1).to(device), dim_id=[3])[0,0,0]

    # delta clamp
    delta_max = 0.05
    delta = u_first - u_prev
    delta_clamped = torch.clamp(delta, -delta_max, delta_max)
    u_first_limited = u_prev + delta_clamped

    u_applied = float(inverse_normalize_x(u_first_limited, dim_id=[3]))

    # Simulate one step
    x_current, depth_current = GAMMA_obj.run_sim_interval(u_applied)

    # Update past sequence
    GAMMA_obj.x_past[:, :-1] = GAMMA_obj.x_past[:, 1:]
    GAMMA_obj.x_past[0, -1] = x_current
    GAMMA_obj.x_past[1, -1] = depth_current

    GAMMA_obj.u_past[:-1] = GAMMA_obj.u_past[1:].clone()
    GAMMA_obj.u_past[-1] = u_applied

    # Save state
    GAMMA_obj.x_hat_current = torch.tensor([x_current, depth_current], device=device)
    GAMMA_obj.x_sys_current = torch.tensor([[x_current], [depth_current]], device=device)
    GAMMA_obj.MPC_counter += 1

    new_state = torch.tensor([[x_current, depth_current]], device=GAMMA_obj.x_past_save.device)
    GAMMA_obj.x_past_save = torch.cat((GAMMA_obj.x_past_save, new_state), dim=0)

    new_u = torch.tensor([[u_applied]], device=GAMMA_obj.u_past_save.device)
    GAMMA_obj.u_past_save = torch.cat((GAMMA_obj.u_past_save, new_u), dim=0)

In [6]:
import os
import matplotlib.pyplot as plt
import numpy as np

def plot_fig(MPC_GAMMA, N_step, save_path=None):
    plt.figure(figsize=[8, 6])

    plt.subplot(3, 1, 1)
    plt.plot(MPC_GAMMA.x_past_save[:N_step, 0].detach().cpu().numpy(), label="GAMMA simulation")
    plt.plot(MPC_GAMMA.ref[:N_step].detach().cpu().numpy(), label="Reference")
    plt.legend()
    plt.xlabel("MPC time step")
    plt.ylabel("Melt Pool Temperature (K)")

    plt.subplot(3, 1, 2)
    plt.plot(MPC_GAMMA.x_past_save[:N_step, 1].detach().cpu().numpy(), label="GAMMA simulation")
    plt.plot(np.arange(N_step), 0.225 * np.ones(N_step), linestyle='--', label="Upper Bound")
    plt.plot(np.arange(N_step), 0.075 * np.ones(N_step), linestyle='--', label="Lower Bound")
    plt.xlabel("MPC time step")
    plt.ylabel("Melt Pool Depth (mm)")
    plt.legend()

    plt.subplot(3, 1, 3)
    plt.plot(MPC_GAMMA.u_past_save[:N_step].detach().cpu().numpy())
    plt.ylabel("Laser Power (W)")
    plt.xlabel("MPC time step")

    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        plt.close()
        print(f"Saved plot to {save_path}")
    else:
        plt.show()


## run loop

In [7]:
import os, torch

# ===== 여기서 device 선택 =====
GPU_ID = 7          # 쓸 GPU 번호 (0,1,2,...). CPU로 쓰려면 -1

if GPU_ID >= 0 and torch.cuda.is_available():
    device = torch.device(f"cuda:{GPU_ID}")
    torch.cuda.set_device(GPU_ID)
    print(f"using device: {device} ({torch.cuda.get_device_name(GPU_ID)})")
else:
    device = torch.device("cpu")
    print("using device: cpu")

print("device_count =", torch.cuda.device_count())
print("current_device =", torch.cuda.current_device() if torch.cuda.is_available() else None)

using device: cuda:7 (NVIDIA RTX A6000)
device_count = 8
current_device = 7


In [8]:
# print(f"device: {device}")
# print(f"model device: {next(model.parameters()).device}")
# print(f"x_min device: {x_min.device}")
# print(f"y_min device: {y_min.device}")
# print(f"init_avg device: {init_avg.device}")
# print(f"fix_covariates device: {fix_covariates.device}")
# print(f"mp_temp_ref device: {mp_temp_ref.device}")

In [9]:
# import os
# import glob
# import pickle

# # ── Constants ─────────────────────────────────────
# INPUT_DATA_DIR = "data"
# SIM_DIR_NAME = "single_track_square"
# BASE_LASER_FILE_DIR = "laser_power_profiles/csv"
# CLOUD_TARGET_BASE_PATH = "result"
# solidus_temp = 1600
# window = 50
# sim_interval = 5
# init_runs = 50
# P = 50

# # laser 1번만 실행
# LASER_NUM = 1
# LASER_CSV_DIR = "/home/ftk3187/github/DPC_research/02_DED/8_policy_0302/split_by_laser_power_number"

# # 학습 노트북(01_train_policy)이 저장한 실험 폴더들의 root
# MODELS_BASE = "/home/ftk3187/github/DPC_research/02_DED/8_policy_0302/models_saved_0624"

# # ── 어떤 .pth 를 돌릴지 선택 ─────────────────────────
# # 기본: 모든 체크포인트 (init / rollout / best 전부)
# PTH_GLOB = "**/policy_roll_epoch*.pth"
# # 각 실험의 best 만 돌리려면:        PTH_GLOB = "**/policy_model_best.pth"
# # rollout 체크포인트만 돌리려면:     PTH_GLOB = "**/policy_roll_epoch*.pth"

# # 중간 진행 플롯을 화면에 띄울지 (sweep이 길면 False 권장; 최종 플롯은 항상 저장됨)
# SHOW_PROGRESS_PLOT = False

# pth_list = sorted(glob.glob(os.path.join(MODELS_BASE, PTH_GLOB), recursive=True))
# print(f"Found {len(pth_list)} checkpoint(s) under:\n  {MODELS_BASE}\n  pattern={PTH_GLOB}")
# for p in pth_list:
#     print("  -", os.path.relpath(p, MODELS_BASE))

# # ── Loop over all checkpoints (laser 1 only) ────────
# for pth_idx, model_path in enumerate(pth_list, start=1):
#     exp_name = os.path.basename(os.path.dirname(model_path))          # 실험 폴더명
#     ckpt_name = os.path.splitext(os.path.basename(model_path))[0]     # 체크포인트명
#     save_dir = os.path.join(os.path.dirname(model_path), "laser1_eval", ckpt_name)

#     print(f"\n========== [{pth_idx}/{len(pth_list)}] {exp_name} / {ckpt_name} ==========")

#     try:
#         os.makedirs(save_dir, exist_ok=True)

#         # ── 모델 구조: 같은 실험 폴더의 policy_parameters.pkl 에서 읽고, 없으면 기본값 ──
#         hidden_dim, n_layers = 1024, 3
#         param_pkl = os.path.join(os.path.dirname(model_path), "policy_parameters.pkl")
#         if os.path.exists(param_pkl):
#             try:
#                 with open(param_pkl, "rb") as f:
#                     _p = pickle.load(f)
#                 hidden_dim = int(_p.get("hidden_dim", hidden_dim))
#                 n_layers = int(_p.get("n_layers", n_layers))
#             except Exception as e:
#                 print(f"[WARN] could not read {param_pkl}: {e} -> use defaults ({n_layers}L,{hidden_dim}H)")

#         # ── Load model ─────────────────────────────────────
#         model = PolicyNN(
#             past_input_dim=6,
#             future_input_dim=6,
#             output_dim=1,
#             p=P,
#             window=window,
#             hidden_dim=hidden_dim,
#             n_layers=n_layers,
#             dropout_p=0.0
#         ).to(device)

#         state_dict = torch.load(model_path, map_location=device)
#         model.load_state_dict(state_dict)
#         model.eval()

#         # ── laser 1 실행 ───────────────────────────────────
#         laser_num = LASER_NUM

#         # Step 1: Load input CSV
#         csv_path = os.path.join(LASER_CSV_DIR, f"laser_power_number_{laser_num}.csv")
#         df = pd.read_csv(csv_path)

#         # Step 2: Instantiate GAMMA class
#         GAMMA_class = GAMMA_obj(INPUT_DATA_DIR, SIM_DIR_NAME, BASE_LASER_FILE_DIR,
#                                 CLOUD_TARGET_BASE_PATH, solidus_temp, window,
#                                 init_runs, sim_interval, laser_power_number=laser_num)

#         # Step 3: Run initial steps
#         init_avg = GAMMA_class.run_initial_steps()
#         init_avg = torch.tensor(init_avg, dtype=torch.float32).to(device)[:, -window:]

#         # Step 4: Prepare inputs
#         loc_Z = df["Z"].to_numpy().reshape(-1, 1)
#         dist_X = df["Dist_to_nearest_X"].to_numpy().reshape(-1, 1)
#         dist_Y = df["Dist_to_nearest_Y"].to_numpy().reshape(-1, 1)
#         fix_covariates = torch.tensor(np.concatenate((loc_Z, dist_X, dist_Y), axis=1), dtype=torch.float32).to(device)

#         laser_power_ref = torch.tensor(df["Laser_power"].to_numpy().reshape(-1, 1), dtype=torch.float32).to(device)
#         laser_power_past = laser_power_ref[:window]

#         mp_temp_raw = df["melt_pool_temperature"].to_numpy()
#         mp_temp = copy.deepcopy(mp_temp_raw)
#         mp_temp[1:-2] = moving_average_1d(mp_temp_raw, 4)
#         mp_temp_ref = torch.tensor(mp_temp, dtype=torch.float32).to(device)

#         # ---- depth는 있으면 만들고, 없으면 스킵 ----
#         if "melt_pool_depth" in df.columns:
#             mp_depth_raw = df["melt_pool_depth"].to_numpy()
#             mp_depth = copy.deepcopy(mp_depth_raw)
#             mp_depth[1:-2] = moving_average_1d(mp_depth_raw, 4)
#             mp_depth_ref = torch.tensor(mp_depth, dtype=torch.float32).to(device)
#         else:
#             mp_depth_ref = None
#             print(f"[WARN] laser {laser_num}: 'melt_pool_depth' column not found -> skip depth.")

#         # Step 5: Initialize GAMMA_class variables
#         GAMMA_class.ref = mp_temp_ref.clone().to(device)
#         GAMMA_class.fix_cov_all = fix_covariates.clone().to(device)
#         GAMMA_class.x_past = init_avg.clone().to(device)
#         GAMMA_class.u_past = laser_power_past.clone().to(device)
#         GAMMA_class.x_past_save = GAMMA_class.x_past.T.clone().to(device)
#         GAMMA_class.u_past_save = GAMMA_class.u_past.clone().to(device)
#         GAMMA_class.MPC_counter = window
#         GAMMA_class.x_hat_current = GAMMA_class.x_past[:, -1].to(device)
#         GAMMA_class.x_sys_current = GAMMA_class.x_past[:, -1].reshape(2, 1).to(device)
#         GAMMA_class.save_time = []

#         # Step 6: Run simulation loop
#         N_step = len(mp_temp_ref) - P
#         for i in tqdm(range(N_step - P), desc=f"{ckpt_name} | Laser {laser_num}"):
#             run_one_step_policy(GAMMA_class, model, P=P, window=window)
#             if SHOW_PROGRESS_PLOT and (i % 500 == 0):
#                 plot_fig(GAMMA_class, N_step)   # 중간 출력(화면)

#         # 최종 플롯만 save_dir에 저장
#         final_plot_path = os.path.join(save_dir, f"final_plot_laser_{laser_num}.png")
#         plot_fig(GAMMA_class, N_step, save_path=final_plot_path)

#         # Step 7: Save outputs
#         x_save = GAMMA_class.x_past_save.detach().cpu().numpy()
#         u_save = GAMMA_class.u_past_save.detach().cpu().numpy()
#         ref_save = GAMMA_class.ref.detach().cpu().numpy()

#         # 시간 저장
#         df_time = pd.DataFrame({'Time': GAMMA_class.save_time})
#         df_time.to_csv(os.path.join(save_dir, f"policy_time_laser_{laser_num}.csv"), index=False)

#         pd.DataFrame(x_save).to_csv(os.path.join(save_dir, f"x_outputs_laser_{laser_num}.csv"), index=False)
#         pd.DataFrame(u_save).to_csv(os.path.join(save_dir, f"u_outputs_laser_{laser_num}.csv"), index=False)
#         pd.DataFrame(ref_save).to_csv(os.path.join(save_dir, f"ref_laser_{laser_num}.csv"), index=False)

#         print(f"✅ Completed: {exp_name}/{ckpt_name} (laser {laser_num}) -> {save_dir}")

#     except Exception as e:
#         print(f"❌ Error in {exp_name}/{ckpt_name}: {e}")


In [ ]:
import os
import glob
import re
import pickle
import time

# ── Constants ─────────────────────────────────────
INPUT_DATA_DIR = "data"
SIM_DIR_NAME = "single_track_square"
BASE_LASER_FILE_DIR = "laser_power_profiles/csv"
CLOUD_TARGET_BASE_PATH = "result"
solidus_temp = 1600
window = 50
sim_interval = 5
init_runs = 50
P = 50

# laser 1번만 실행
LASER_NUM = 1
LASER_CSV_DIR = "/home/ftk3187/github/DPC_research/02_DED/8_policy_0302/split_by_laser_power_number"

# 학습 노트북(01_train_policy)이 저장한 실험 폴더들의 root
MODELS_BASE = "/home/ftk3187/github/DPC_research/02_DED/8_policy_0302/models_saved_0624"

# 어떤 .pth 를 돌릴지
PTH_GLOB = "**/policy_roll_epoch*.pth"
# best 만:    PTH_GLOB = "**/policy_model_best.pth"

EXCLUDE_EPOCHS = {}     # ★ 제외할 에폭


def epoch_of(p):
    """파일명에서 에폭 정수 추출 (.pth 유무·0패딩 모두 허용). best 등은 None."""
    m = re.search(r"epoch0*(\d+)", os.path.basename(p))
    return int(m.group(1)) if m else None


def list_pths():
    """제외 에폭을 뺀 .pth 목록. 모든 glob을 이 함수로 통일한다."""
    found = glob.glob(os.path.join(MODELS_BASE, PTH_GLOB), recursive=True)
    return sorted(p for p in found if epoch_of(p) not in EXCLUDE_EPOCHS)


# 확인용 출력
paths = list_pths()
print(f"{len(paths)} files (epoch {sorted(EXCLUDE_EPOCHS)} 제외)")
for p in paths:
    print(" ", os.path.relpath(p, MODELS_BASE))

SHOW_PROGRESS_PLOT = False  # sweep 길면 False 권장 (최종 플롯은 항상 저장)

# ── Watch 설정 ──────────────────────────────────────
IDLE_SLEEP = 60          # 돌릴 게 없을 때만 이만큼(초) 쉬고 다시 스캔
IDLE_TIMEOUT_MIN = 60    # 이 시간(분) 동안 새 결과가 0개면 종료 (None이면 끔)

# 제외를 반영한 실제 대상 수 (이게 있어야 watch 가 정확히 끝남)
EXPECTED_TOTAL = len(list_pths())


def result_csv_path(model_path):
    """이 체크포인트의 '완료 표식' 파일 경로 (가장 마지막에 쓰는 파일)."""
    ckpt_name = os.path.splitext(os.path.basename(model_path))[0]
    save_dir = os.path.join(os.path.dirname(model_path), "laser1_eval", ckpt_name)
    return os.path.join(save_dir, f"x_outputs_laser_{LASER_NUM}.csv")


def is_done(model_path):
    return os.path.exists(result_csv_path(model_path))


def count_done():
    return sum(1 for p in list_pths() if is_done(p))


def evaluate_checkpoint(model_path):
    """주어진 .pth 로 laser 1 시뮬레이션 후 결과 저장. 성공 시 True."""
    # 이중 안전장치: 어떤 경로로 들어와도 제외 에폭이면 건너뜀
    if epoch_of(model_path) in EXCLUDE_EPOCHS:
        print(f"skip excluded epoch: {os.path.basename(model_path)}")
        return False

    exp_name = os.path.basename(os.path.dirname(model_path))
    ckpt_name = os.path.splitext(os.path.basename(model_path))[0]
    save_dir = os.path.join(os.path.dirname(model_path), "laser1_eval", ckpt_name)
    os.makedirs(save_dir, exist_ok=True)

    print(f"\n========== {exp_name} / {ckpt_name} ==========")

    # ── 모델 구조: 같은 실험 폴더의 policy_parameters.pkl 에서 읽고, 없으면 기본값 ──
    hidden_dim, n_layers = 1024, 3
    param_pkl = os.path.join(os.path.dirname(model_path), "policy_parameters.pkl")
    if os.path.exists(param_pkl):
        try:
            with open(param_pkl, "rb") as f:
                _p = pickle.load(f)
            hidden_dim = int(_p.get("hidden_dim", hidden_dim))
            n_layers = int(_p.get("n_layers", n_layers))
        except Exception as e:
            print(f"[WARN] could not read {param_pkl}: {e} -> use defaults ({n_layers}L,{hidden_dim}H)")

    # ── Load model ─────────────────────────────────────
    model = PolicyNN(
        past_input_dim=6,
        future_input_dim=6,
        output_dim=1,
        p=P,
        window=window,
        hidden_dim=hidden_dim,
        n_layers=n_layers,
        dropout_p=0.0
    ).to(device)

    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    # ── laser 1 실행 ───────────────────────────────────
    laser_num = LASER_NUM

    # Step 1: Load input CSV
    csv_path = os.path.join(LASER_CSV_DIR, f"laser_power_number_{laser_num}.csv")
    df = pd.read_csv(csv_path)

    # Step 2: Instantiate GAMMA class
    GAMMA_class = GAMMA_obj(INPUT_DATA_DIR, SIM_DIR_NAME, BASE_LASER_FILE_DIR,
                            CLOUD_TARGET_BASE_PATH, solidus_temp, window,
                            init_runs, sim_interval, laser_power_number=laser_num)

    # Step 3: Run initial steps
    init_avg = GAMMA_class.run_initial_steps()
    init_avg = torch.tensor(init_avg, dtype=torch.float32).to(device)[:, -window:]

    # Step 4: Prepare inputs
    loc_Z = df["Z"].to_numpy().reshape(-1, 1)
    dist_X = df["Dist_to_nearest_X"].to_numpy().reshape(-1, 1)
    dist_Y = df["Dist_to_nearest_Y"].to_numpy().reshape(-1, 1)
    fix_covariates = torch.tensor(np.concatenate((loc_Z, dist_X, dist_Y), axis=1), dtype=torch.float32).to(device)

    laser_power_ref = torch.tensor(df["Laser_power"].to_numpy().reshape(-1, 1), dtype=torch.float32).to(device)
    laser_power_past = laser_power_ref[:window]

    mp_temp_raw = df["melt_pool_temperature"].to_numpy()
    mp_temp = copy.deepcopy(mp_temp_raw)
    mp_temp[1:-2] = moving_average_1d(mp_temp_raw, 4)
    mp_temp_ref = torch.tensor(mp_temp, dtype=torch.float32).to(device)

    # ---- depth는 있으면 만들고, 없으면 스킵 ----
    if "melt_pool_depth" in df.columns:
        mp_depth_raw = df["melt_pool_depth"].to_numpy()
        mp_depth = copy.deepcopy(mp_depth_raw)
        mp_depth[1:-2] = moving_average_1d(mp_depth_raw, 4)
        mp_depth_ref = torch.tensor(mp_depth, dtype=torch.float32).to(device)
    else:
        mp_depth_ref = None
        print(f"[WARN] laser {laser_num}: 'melt_pool_depth' column not found -> skip depth.")

    # Step 5: Initialize GAMMA_class variables
    GAMMA_class.ref = mp_temp_ref.clone().to(device)
    GAMMA_class.fix_cov_all = fix_covariates.clone().to(device)
    GAMMA_class.x_past = init_avg.clone().to(device)
    GAMMA_class.u_past = laser_power_past.clone().to(device)
    GAMMA_class.x_past_save = GAMMA_class.x_past.T.clone().to(device)
    GAMMA_class.u_past_save = GAMMA_class.u_past.clone().to(device)
    GAMMA_class.MPC_counter = window
    GAMMA_class.x_hat_current = GAMMA_class.x_past[:, -1].to(device)
    GAMMA_class.x_sys_current = GAMMA_class.x_past[:, -1].reshape(2, 1).to(device)
    GAMMA_class.save_time = []

    # Step 6: Run simulation loop
    N_step = len(mp_temp_ref) - P
    for i in tqdm(range(N_step - P), desc=f"{ckpt_name} | Laser {laser_num}"):
        run_one_step_policy(GAMMA_class, model, P=P, window=window)
        if SHOW_PROGRESS_PLOT and (i % 500 == 0):
            plot_fig(GAMMA_class, N_step)   # 중간 출력(화면)

    # 최종 플롯만 save_dir에 저장
    final_plot_path = os.path.join(save_dir, f"final_plot_laser_{laser_num}.png")
    plot_fig(GAMMA_class, N_step, save_path=final_plot_path)

    # Step 7: Save outputs  (x_outputs 를 '맨 마지막'에 써서 완료 표식으로 사용)
    x_save = GAMMA_class.x_past_save.detach().cpu().numpy()
    u_save = GAMMA_class.u_past_save.detach().cpu().numpy()
    ref_save = GAMMA_class.ref.detach().cpu().numpy()

    df_time = pd.DataFrame({'Time': GAMMA_class.save_time})
    df_time.to_csv(os.path.join(save_dir, f"policy_time_laser_{laser_num}.csv"), index=False)
    pd.DataFrame(ref_save).to_csv(os.path.join(save_dir, f"ref_laser_{laser_num}.csv"), index=False)
    pd.DataFrame(u_save).to_csv(os.path.join(save_dir, f"u_outputs_laser_{laser_num}.csv"), index=False)
    pd.DataFrame(x_save).to_csv(os.path.join(save_dir, f"x_outputs_laser_{laser_num}.csv"), index=False)  # ← 마지막

    print(f"✅ Completed: {exp_name}/{ckpt_name} (laser {laser_num}) -> {save_dir}")
    return True

# ── Watch loop (fixed) ─────────────────────────────
WAIT_FOR_NEW = True   # True: 학습이 새 체크포인트 계속 만들면 기다림
                      # False: 지금 있는 것만 다 돌리고 바로 종료

failed = set()
last_progress = time.time()

done = count_done()
print(f"[watch] 시작 완료={done}/{len(list_pths())}")

while True:
    pth_list = list_pths()                                   # 매 루프 새로 스캔 → 새 .pth 반영
    todo = [p for p in pth_list if (not is_done(p)) and (p not in failed)]

    if todo:
        model_path = todo[0]
        try:
            evaluate_checkpoint(model_path)
            last_progress = time.time()                      # ★ 성공할 때만 진행시간 갱신
        except Exception as e:
            print(f"❌ Error in {model_path}: {e}")
            failed.add(model_path)
        continue                                             # 끝나면 바로 재스캔

    # ── 여기는 '지금 돌릴 게 없을 때'만 도달 ──
    done = count_done()
    total = len(pth_list)

    if not WAIT_FOR_NEW:
        print(f"[watch] 남은 작업 없음 → 종료 (완료 {done}/{total}, 실패 {len(failed)})")
        break

    idle_min = (time.time() - last_progress) / 60.0
    if (IDLE_TIMEOUT_MIN is not None) and (idle_min >= IDLE_TIMEOUT_MIN):
        print(f"[watch] {IDLE_TIMEOUT_MIN}분간 새 결과 없음 → 종료 (완료 {done}/{total}, 실패 {len(failed)})")
        break

    print(f"[watch] 대기... 완료 {done}/{total}, idle {idle_min:.1f}m, 실패 {len(failed)} | {IDLE_SLEEP}s sleep")
    time.sleep(IDLE_SLEEP)

done = count_done()
print(f"[watch] DONE. 완료 {done}/{len(list_pths())}, 실패 {len(failed)}개")
if failed:
    print("  실패 목록 (셀 다시 실행하면 재시도):")
    for p in sorted(failed):
        print("   -", os.path.relpath(p, MODELS_BASE))

93 files (epoch [] 제외)
  or_3L_1024H_s1_c10_ep1000_roll100_m51200_final_0426_boosting(t10s10)_lr0.0001_rand(1234)_aug/policy_roll_epoch0099.pth
  or_3L_1024H_s1_c10_ep1000_roll100_m51200_final_0426_boosting(t10s10)_lr0.0001_rand(1234)_aug/policy_roll_epoch0199.pth
  or_3L_1024H_s1_c10_ep1000_roll100_m51200_final_0426_boosting(t10s10)_lr0.0001_rand(1234)_aug/policy_roll_epoch0299.pth
  or_3L_1024H_s1_c10_ep1000_roll100_m51200_final_0426_boosting(t10s10)_lr0.0001_rand(1234)_aug/policy_roll_epoch0399.pth
  or_3L_1024H_s1_c10_ep1000_roll100_m51200_final_0426_boosting(t10s10)_lr0.0001_rand(1234)_aug/policy_roll_epoch0499.pth
  or_3L_1024H_s1_c10_ep100_roll10_m102400_final_0426_boosting(t10s10)_lr0.0001_rand(1234)_aug/policy_roll_epoch0009.pth
  or_3L_1024H_s1_c10_ep100_roll10_m102400_final_0426_boosting(t10s10)_lr0.0001_rand(1234)_aug/policy_roll_epoch0019.pth
  or_3L_1024H_s1_c10_ep100_roll10_m102400_final_0426_boosting(t10s10)_lr0.0001_rand(1234)_aug/policy_roll_epoch0029.pth
  or_3L_1024

100%|██████████| 250/250 [00:05<00:00, 42.90it/s]


[WARN] laser 1: 'melt_pool_depth' column not found -> skip depth.


policy_roll_epoch0299 | Laser 1:   3%|▎         | 159/6195 [00:19<11:51,  8.48it/s]